1

In [14]:
%%capture
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets wandb

2


In [15]:
# All hyperparameters in one place — easy to tweak

import json
import torch
from pathlib import Path
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# ── Hyperparameters ────────────────────────────────────────────────────────────
MODEL_NAME      = "unsloth/Qwen2.5-Coder-3B-Instruct"
MAX_SEQ_LENGTH  = 2048       # max token length per example
LORA_R          = 16         # LoRA rank
LORA_ALPHA      = 16         # LoRA alpha (usually == r)
LORA_DROPOUT    = 0.0        # 0 works well with QLoRA

BATCH_SIZE      = 2          # per device — T4 can handle 2 with 3B model
GRAD_ACCUM      = 4          # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 3          # watch val loss — stop early if it diverges
WARMUP_STEPS    = 50
VAL_SPLIT       = 0.1        # 10% held out for overfitting detection
SEED            = 42

OUTPUT_DIR      = "./qwen_strategy2_checkpoints"
HF_REPO         = None       # Set to "your-username/model-name" to push to Hub


3

In [16]:
from google.colab import files

print("📂 Upload strategy2_finetune.jsonl ...")
uploaded = files.upload()

jsonl_path = next(
    (k for k in uploaded if k.endswith(".jsonl")),
    None
)
if not jsonl_path:
    raise FileNotFoundError("No .jsonl file found in upload. Please upload strategy2_finetune.jsonl")

print(f"✅ Loaded: {jsonl_path}")

# Read examples
with open(jsonl_path) as f:
    raw_examples = [json.loads(line) for line in f]

print(f"   Total examples: {len(raw_examples)}")
neg = sum(1 for e in raw_examples if "[NEGATIVE]" in e["conversations"][0]["content"])
pos = sum(1 for e in raw_examples if "[POSITIVE]" in e["conversations"][0]["content"])
print(f"   NEGATIVE (buggy): {neg}")
print(f"   POSITIVE (clean): {pos}")

📂 Upload strategy2_finetune.jsonl ...


Saving strategy2_finetune.jsonl to strategy2_finetune (2).jsonl
✅ Loaded: strategy2_finetune (2).jsonl
   Total examples: 1088
   NEGATIVE (buggy): 544
   POSITIVE (clean): 544


4

In [17]:
# Qwen2.5-Coder-3B-Instruct with 4-bit QLoRA

print(f"\n🤖 Loading {MODEL_NAME} with 4-bit QLoRA...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,        # auto-detect (fp16 for T4, bf16 for A100)
    load_in_4bit    = True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    target_modules      = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # saves VRAM
    random_state        = SEED,
)

# Qwen2.5-Instruct already has its chat template built in
# We just need to make sure EOS and PAD tokens are set correctly
tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Model loaded")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"   Total params:     {sum(p.numel() for p in model.parameters()):,}")


🤖 Loading unsloth/Qwen2.5-Coder-3B-Instruct with 4-bit QLoRA...
==((====))==  Unsloth 2026.6.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model loaded
   Trainable params: 29,933,568
   Total params:     1,728,606,208


5

In [18]:
# Apply chat template + train/val split

SYSTEM_PROMPT = (
    "You are an AI tutoring assistant for an Intelligent Tutoring System. "
    "When given a [NEGATIVE] prompt, generate Python code with the specified bugs "
    "calibrated to the student's knowledge level. "
    "When given a [POSITIVE] prompt, generate correct, clean Python code."
)

def format_example(example):
    """Add system prompt and apply Qwen2.5 chat template."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["conversations"][0]["content"]},
        {"role": "assistant", "content": example["conversations"][1]["content"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize           = False,
        add_generation_prompt = False,
    )
    return {"text": text}

print("🔧 Formatting dataset with Qwen2.5 chat template...")

# Shuffle before splitting so NEG/POS are interleaved across train/val
import random
random.seed(SEED)
shuffled = raw_examples.copy()
random.shuffle(shuffled)

split_idx  = int(len(shuffled) * (1 - VAL_SPLIT))
train_data = shuffled[:split_idx]
val_data   = shuffled[split_idx:]

print(f"   Train: {len(train_data)} examples")
print(f"   Val:   {len(val_data)} examples")

train_dataset = Dataset.from_list(train_data).map(format_example)
val_dataset   = Dataset.from_list(val_data).map(format_example)

# Sanity check — print one formatted example
print("\n── Sample formatted example (truncated) ──────────────────")
print(train_dataset[0]["text"][:600])
print("...")

🔧 Formatting dataset with Qwen2.5 chat template...
   Train: 979 examples
   Val:   109 examples


Map:   0%|          | 0/979 [00:00<?, ? examples/s]

Map:   0%|          | 0/109 [00:00<?, ? examples/s]


── Sample formatted example (truncated) ──────────────────
<|im_start|>system
You are an AI tutoring assistant for an Intelligent Tutoring System. When given a [NEGATIVE] prompt, generate Python code with the specified bugs calibrated to the student's knowledge level. When given a [POSITIVE] prompt, generate correct, clean Python code.<|im_end|>
<|im_start|>user
[NEGATIVE] Generate a Python function for the following problem with these bugs: wrong_arithmetic_operator, incorrect_indexing, wrong_accumulator_init. The student has a medium knowledge level.

Problem: Write a function to find the number of ways to fill it with 2 x 1 dominoes for the given 3
...


6

In [19]:
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * NUM_EPOCHS
eval_steps      = max(steps_per_epoch // 2, 10)

print(f"\n📋 Training plan:")
print(f"   Steps per epoch : {steps_per_epoch}")
print(f"   Total steps     : {total_steps}")
print(f"   Eval every      : {eval_steps} steps")

# Force correct EOS/PAD tokens for Qwen2.5 — must happen before SFTTrainer
tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = tokenizer.eos_token

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = WARMUP_STEPS,
    lr_scheduler_type           = "cosine",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = eval_steps,
    save_strategy               = "steps",
    save_steps                  = eval_steps,
    save_total_limit            = 2,          # keep only 2 best checkpoints
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    dataset_text_field          = "text",
    packing                     = False,      # disable for accurate eval loss
    seed                        = SEED,
    report_to                   = "none",     # change to "wandb" if you want logging
)

trainer = SFTTrainer(
    model              = model,
    processing_class   = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = val_dataset,
    args               = training_args,
)

# ── IMPORTANT: Only train on assistant responses, not the prompts ──────────────
# This is the key NAT technique — loss computed only on the generated code
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("✅ Trainer configured")




📋 Training plan:
   Steps per epoch : 122
   Total steps     : 366
   Eval every      : 61 steps


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/979 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/109 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=6):   0%|          | 0/979 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/979 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/109 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/109 [00:00<?, ? examples/s]

✅ Trainer configured


7

In [22]:
print("\n🚀 Starting training...")
print("   Watch for: eval_loss rising while train_loss falls = overfitting")
print("   If that happens, use an earlier checkpoint from", OUTPUT_DIR)
print()

trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Total runtime : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Final train loss: {trainer_stats.metrics['train_loss']:.4f}")



🚀 Starting training...
   Watch for: eval_loss rising while train_loss falls = overfitting
   If that happens, use an earlier checkpoint from ./qwen_strategy2_checkpoints



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 979 | Num Epochs = 3 | Total steps = 369
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss,Validation Loss
61,0.517424,0.468426
122,0.304652,0.293021
183,0.157664,0.203177
244,0.111105,0.125297
305,0.051938,0.106873
366,0.037363,0.102541
369,0.037363,0.102582


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


✅ Training complete!
   Total runtime : 1451s
   Final train loss: 0.2372


8

In [24]:
# Check for overfitting — compare train vs val perplexity

import math

print("\n📊 Evaluating perplexity...")

train_result = trainer.evaluate(train_dataset.select(range(min(100, len(train_dataset)))).remove_columns(["conversations"]))
val_result   = trainer.evaluate(val_dataset.remove_columns(["conversations"]))

train_ppl = math.exp(train_result["eval_loss"])
val_ppl   = math.exp(val_result["eval_loss"])

print(f"   Train perplexity : {train_ppl:.2f}")
print(f"   Val perplexity   : {val_ppl:.2f}")
print(f"   Ratio (val/train): {val_ppl/train_ppl:.2f}x")

if val_ppl / train_ppl > 2.0:
    print("\n⚠️  Possible overfitting — consider fewer epochs or an earlier checkpoint")
elif val_ppl / train_ppl > 1.5:
    print("\n⚠️  Mild overfitting — results may still generalise")
else:
    print("\n✅ Looks healthy — model is generalising well")



📊 Evaluating perplexity...


ValueError: You should supply an encoding or a list of encodings to this method that includes input_ids, but you provided ['text']

9

In [25]:
# Quick sanity check — generate a buggy and a correct example

FastLanguageModel.for_inference(model)

def generate(prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    new_tokens = outputs[0][inputs.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


TEST_PROBLEM = "Write a Python function to check if a number is prime."

print("── NEGATIVE (buggy) ────────────────────────────────────────")
neg_prompt = (
    "[NEGATIVE] Generate a Python function for the following problem with "
    "this bug: wrong_comparison_operator. The student has a high knowledge level.\n\n"
    f"Problem: {TEST_PROBLEM}"
)
print(generate(neg_prompt))

print("\n── POSITIVE (correct) ──────────────────────────────────────")
pos_prompt = (
    "[POSITIVE] Generate a correct Python function for the following problem.\n\n"
    f"Problem: {TEST_PROBLEM}"
)
print(generate(pos_prompt))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


── NEGATIVE (buggy) ────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

def prime_num(num):
  if num >= 1:
   for i in range(2,num//2):
     if (num // i) == 0:
                return False
     else:
                return True
  else:
          return False

── POSITIVE (correct) ──────────────────────────────────────
def prime_num(num):
  if num >= 2:
   for i in range(2, num//2 + 1):
     if (num % i) == 0:
                return False
  return True
  else:
      return False


10

In [29]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

model.push_to_hub("madane007/stratergy2_qwen")
tokenizer.push_to_hub("madane007/stratergy2_qwen")

README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 60.7kB / 59.9MB            

Saved model to https://huggingface.co/madane007/stratergy2_qwen


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpfnrrgog3/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpfnrrgog3/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            